In [1]:
!pip install langgraph>=0.0.10 \
    langchain-anthropic>=0.1.1 \
    langchain-core>=0.1.27 \
    PyGithub>=2.2.0 \
    GitPython>=3.1.42 \
    pydantic>=2.6.1 \
    python-dotenv>=1.0.0

In [2]:
from typing import TypedDict, Optional
from langgraph.graph import StateGraph, END, START
from langchain_core.prompts import ChatPromptTemplate
from langchain_anthropic import ChatAnthropic
from pydantic import BaseModel, Field
from git import Repo
from github import Github
import os

class CodeSolution(BaseModel):
    """Schema for code solutions."""
    description: str = Field(description="Description of the solution approach")
    code: str = Field(description="Complete code including imports and docstring")
    branch_name: str = Field(description="Suggested branch name for the new feature or fix (e.g., 'feat/array-products-calculator').")
    file_path: str = Field(description="Path relative to the repository root where the code should be saved (e.g., 'array_products.py').")
    commit_message: str = Field(description="Commit message for the changes (e.g., 'feat: add array products calculator').")
    pr_title: str = Field(description="Title for the pull request (e.g., 'Add Array Products Calculator').")
    pr_description: str = Field(description="Detailed description for the pull request, including the solution approach and any relevant context.")

class GraphState(TypedDict, total=False):
    """State for the task processing workflow."""
    status: str  # Using status instead of error to track state
    task_content: str
    repo_dir: str
    generation: CodeSolution | None
    iterations: int
    error_message: Optional[str]
    repo_name: str
    pr_url: str
    github_token: str # Added github_token to the state definition

def clone_repository(github_token: str, repo_name: str) -> tuple[str, str]:
    """Clone the repository and return the local directory."""
    try:
        # In Colab, __file__ is not defined, so use os.getcwd() for the current directory.
        script_dir = os.getcwd()
        repo_dir = os.path.join(script_dir, 'agent-task')

        if os.path.exists(repo_dir):
            os.system(f'rm -rf {repo_dir}')

        g = Github(auth=Auth.Token(github_token)) # Use Auth.Token to avoid DeprecationWarning
        # Ensure the repo_name is in the correct format for get_repo (owner/repo)
        # The user provided 'eelizan1/agent-task', which is correct.
        repo = g.get_repo(repo_name)
        Repo.clone_from(repo.clone_url, repo_dir)
        return repo_dir, ""
    except Exception as e:
        return "", f"Repository setup failed: {str(e)}"

def read_task_file(repo_dir: str) -> tuple[str, str]:
    """Read the task markdown file."""
    try:
        # Correcting the path for task.md based on the file listing
        task_path = os.path.join(repo_dir, 'task.md')
        with open(task_path, 'r') as f:
            return f.read(), ""
    except Exception as e:
        return "", f"Failed to read task: {str(e)}"

def initialize_state(github_token: str, repo_name: str) -> dict:
    """Initialize the workflow state."""
    repo_dir, error = clone_repository(github_token, repo_name)
    if error:
        return {
            "status": "failed",
            "task_content": "",
            "repo_dir": "",
            "generation": None,
            "iterations": 0,
            "error_message": error
        }

    task_content, error = read_task_file(repo_dir)
    if error:
        return {
            "status": "failed",
            "task_content": "",
            "repo_dir": repo_dir,
            "generation": None,
            "iterations": 0,
            "error_message": error
        }

    return {
        "status": "ready",
        "task_content": task_content,
        "repo_dir": repo_dir,
        "generation": None,
        "iterations": 0,
        "error_message": None
    }

def generate_solution(state: GraphState):
    """Generate code solution based on task description."""
    # Increment iterations for this attempt regardless of prior status
    current_iterations = state.get("iterations", 0) + 1
    print(f"Generating solution - Attempt #{current_iterations}")

    task_content = state["task_content"]

    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are a Python developer. Generate a solution based on the task requirements.
        Include complete code with imports, type hints, docstring, and examples.
        Also provide a suggested branch name, file path relative to the repository root, a concise commit message,
        a title for the pull request, and a detailed description for the pull request.
        The file path should be a Python file, e.g., 'src/my_solution.py'."""),
        ("human", "Task description:\n{task}"),
    ])

    try:
        llm = ChatAnthropic(
            model="claude-opus-4-6",
            temperature=0,
            anthropic_api_key=ANTHROPIC_API_KEY
        )
        chain = prompt | llm.with_structured_output(CodeSolution)

        solution = chain.invoke({"task": task_content})

        # Explicit validation of required attributes after generation
        for attr in ['branch_name', 'file_path', 'commit_message', 'pr_title', 'pr_description']:
            if not hasattr(solution, attr) or getattr(solution, attr) is None or getattr(solution, attr) == '':
                raise ValueError(f"Generated CodeSolution missing or empty required attribute: '{attr}'")

        return {
            **state, # Preserve existing state attributes
            "status": "generated",
            "generation": solution,
            "iterations": current_iterations,
            "error_message": None
        }
    except Exception as e:
        print(f"Generation failed on attempt #{current_iterations}: {e}")
        return {
            **state,
            "status": "failed",
            "iterations": current_iterations,
            "error_message": f"Solution generation failed: {str(e)}"
        }

def test_solution(state: GraphState):
    """Test the generated code solution."""
    if state["status"] != "generated" or not state["generation"]:
        return {
            **state,
            "status": "failed",
            "error_message": state.get("error_message") or "Cannot test solution: not generated or no generation found."
        }

    try:
        namespace = {}
        # Execute the generated code dynamically
        exec(state["generation"].code, namespace)

        # Ensure the 'uncompress' function exists
        if 'uncompress' not in namespace:
            return {**state, "status": "failed", "error_message": "Test failed: 'uncompress' function not found in generated code."}

        # Test cases for the uncompress function
        test_cases = [
            ("2c3a1t", "ccaaat"),
            ("4s2b", "ssssbb"),
            ("2p1o5p", "ppoppppp"),
            ("3n12e2z", "nnneeeeeeeeeeeezz"),
            ("127y", "yyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyyy")
        ]

        for input_str, expected_output in test_cases:
            result = namespace['uncompress'](input_str)
            if result != expected_output:
                return {**state, "status": "failed", "error_message": f"Test failed for input '{input_str}': Expected '{expected_output}', got '{result}'"}

        return {**state, "status": "tested", "error_message": None}

    except Exception as e:
        return {**state, "status": "failed", "error_message": f"Test execution failed: {str(e)}"}

from github import Github, Auth
from git import Repo

def create_pr(state: GraphState):
    try:
        repo = Repo(state["repo_dir"])
        github_token = state["github_token"]
        repo_name = state["repo_name"]
        branch_name = state["generation"].branch_name

        auth_url = f"https://{github_token}@github.com/{repo_name}.git"
        repo.remotes.origin.set_url(auth_url)

        if branch_name not in repo.heads:
            repo.git.checkout("-b", branch_name)
        else:
            repo.git.checkout(branch_name)

        file_path = os.path.join(state["repo_dir"], state["generation"].file_path)
        with open(file_path, "w") as f:
            f.write(state["generation"].code)

        repo.git.add(A=True)
        repo.index.commit(state["generation"].commit_message)

        repo.remotes.origin.push(branch_name)

        g = Github(auth=Auth.Token(github_token))
        gh_repo = g.get_repo(repo_name)

        pr = gh_repo.create_pull(
            title=state["generation"].pr_title,
            body=state["generation"].pr_description,
            head=branch_name,
            base="main"
        )

        print(f"Created PR: {pr.html_url}")

        return {
            **state,
            "status": "completed",
            "pr_url": pr.html_url
        }

    except Exception as e:
        print(f"PR creation failed: {e}")
        return {
            **state,
            "status": "failed",
            "error_message": f"PR creation failed: {str(e)}"
        }

def after_generate(state: GraphState) -> str:
    """Determine next step based on status after generation."""
    if state["status"] == "generated":
        return "test"
    if state["iterations"] < 3:
        return "generate"
    return "end"

def should_continue(state: GraphState) -> str:
    """Determine next step after testing."""
    if state["status"] == "tested":
        return "continue"
    if state["status"] == "failed" and state["iterations"] < 3:
        return "generate"
    return "end"

def create_agent(github_token: str, repo_name: str):
    workflow = StateGraph(GraphState)
    workflow.add_node("generate", generate_solution)
    workflow.add_node("test", test_solution)
    workflow.add_node("create_pr", create_pr)

    workflow.add_edge(START, "generate")

    workflow.add_conditional_edges(
        "generate",
        after_generate,
        {
            "test": "test",
            "generate": "generate",
            "end": END
        }
    )

    workflow.add_conditional_edges(
        "test",
        should_continue,
        {
            "generate": "generate",
            "continue": "create_pr",
            "end": END
        }
    )

    workflow.add_edge("create_pr", END)
    return workflow.compile()

def run_agent(github_token: str, repo_name: str):
    """Run the agent to generate and submit a solution."""
    try:
        agent = create_agent(github_token, repo_name)
        initial_state = initialize_state(github_token, repo_name)
        # Pass repo_name to the state for use in create_pr
        initial_state['repo_name'] = repo_name
        initial_state['github_token'] = github_token # Added github_token to initial_state

        if initial_state["status"] == "failed":
            print(f"Failed to initialize agent: {initial_state.get('error_message', 'Unknown error during initialization')}")
            return {"status": "failed"}

        result = agent.invoke(initial_state)

        if result["status"] == "completed":
            print("Successfully created PR with solution!")
        else:
            print(f"Failed to create solution: {result.get('error_message', 'No specific error message provided.')}")

        return {
            "status": result["status"],
            "generation": result["generation"].code if result["generation"] else None,
            "pr_url": result.get("pr_url"),
            "error_message": result.get("error_message")
        }

    except Exception as e:
        print(f"Agent execution failed unexpectedly: {str(e)}")
        return {"status": "failed", "error_message": f"Agent execution failed unexpectedly: {str(e)}"}

In [3]:

import os
import getpass
from dotenv import load_dotenv
load_dotenv()

os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("ANTHROPIC_API_KEY: ")
os.environ["GITHUB_TOKEN"] = getpass.getpass("GITHUB_TOKEN: ")

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

if not GITHUB_TOKEN:
    raise ValueError("GITHUB_TOKEN is not set")
if not ANTHROPIC_API_KEY:
    raise ValueError("ANTHROPIC_API_KEY is not set")

REPO_NAME = "eelizan1/agent-task-uncompress"

ANTHROPIC_API_KEY: ··········
GITHUB_TOKEN: ··········


In [4]:
result = run_agent(GITHUB_TOKEN, REPO_NAME)

if result["status"] == "completed":
    print("\nSolution generated successfully!")
    if result.get("pr_url"):
        print(f"\nPull Request created at: {result['pr_url']}")
    if result.get("generation"):
        print("\nGenerated Code:")
        print(result["generation"])
else:
    print("\nFailed to generate solution")

Generating solution - Attempt #1
Created PR: https://github.com/eelizan1/agent-task-uncompress/pull/1
Successfully created PR with solution!

Solution generated successfully!

Pull Request created at: https://github.com/eelizan1/agent-task-uncompress/pull/1

Generated Code:
"""Module for uncompressing run-length encoded strings."""

import re


def uncompress(s: str) -> str:
    """Uncompress a run-length encoded string.

    Takes a string formatted as consecutive groups of `<number><char>` and
    returns the uncompressed version where each character is repeated the
    specified number of times.

    Args:
        s: A compressed string consisting of one or more `<number><char>` groups.
           The number may be one or more digits. The char is a single alphabetic
           character.

    Returns:
        The uncompressed string with each character repeated according to its
        preceding number.

    Raises:
        ValueError: If the input string is empty or malformed (does